# Classical ML models (KNN, RandomForest, SVM)

This notebook loads precomputed features from the `dataset/*/input/...` folders, builds a feature matrix by concatenating compound and protein features for each interaction, trains KNN / RandomForest / SVM models, evaluates them (AUC, accuracy), and saves trained models and results to `output/`.

Assumptions: `interactions.npy` is an (N,3) array where each row is `[compound_index, protein_index, label]`.

In [74]:
# Imports
import os
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
import joblib
from collections import defaultdict
print('libraries loaded')

libraries loaded


In [75]:
def load_interaction_dataset(input_dir):
    """Load compounds, proteins and interactions from `input_dir`.
    Supports two formats for `interactions.npy`:
      1) label-only: shape (N,) or (N,1) containing 0/1 labels where
         `compounds` and `proteins` are parallel arrays of length N (one
         compound feature vector and one protein feature vector per interaction).
      2) index-triplets: numeric array shape (N,3) with [compound_idx, protein_idx, label].

    Returns X (N, Dc+Dp), y (N,)
    """
    input_dir = os.path.abspath(input_dir)
    cp_file = os.path.join(input_dir, 'compounds.npy')
    if not os.path.exists(cp_file):
        cp_file = os.path.join(input_dir, 'compounds_old.npy')
    prot_file = os.path.join(input_dir, 'proteins.npy')
    int_file = os.path.join(input_dir, 'interactions.npy')

    if not os.path.exists(cp_file) or not os.path.exists(prot_file) or not os.path.exists(int_file):
        missing = [p for p in (cp_file, prot_file, int_file) if not os.path.exists(p)]
        raise FileNotFoundError(f'Missing files in {input_dir}: {missing}')

    try:
        compounds = np.load(cp_file, allow_pickle=True)
        proteins = np.load(prot_file, allow_pickle=True)
        interactions = np.load(int_file, allow_pickle=True)

        print("compounds.shape : ", compounds.shape, "proteins.shape : ", proteins.shape, "interactions.shape : ", interactions.shape)
    except Exception as e:
        raise RuntimeError(f'Error loading .npy files from {input_dir}: {e}')

    # Robust converter: convert a 1-D object array of vector-like items into a
    # numeric 2-D array. If inner vectors have varying lengths, pad with zeros.
    def ensure_2d_numeric(arr, name):
        # If already numeric 2-D, just cast to float
        if getattr(arr, 'ndim', None) == 2 and arr.dtype != object:
            return arr.astype(float)
        # If 1-D, try to convert each element to a 1-D ndarray
        if getattr(arr, 'ndim', None) == 1:
            lst = []
            for i, elem in enumerate(arr):
                try:
                    a = np.asarray(elem)
                    if a.ndim == 0:
                        # scalar -> make length-1
                        a = a.reshape(1)
                    lst.append(a.ravel())
                except Exception:
                    raise ValueError(f'Element {i} of `{name}` cannot be converted to numeric array; type={type(elem)}')
            lengths = [x.shape[0] for x in lst]
            uniq = sorted(set(lengths))
            if len(uniq) == 1:
                stacked = np.vstack(lst).astype(float)
                return stacked
            # varying lengths: pad with zeros to maxlen
            maxlen = max(lengths)
            stacked = np.zeros((len(lst), maxlen), dtype=float)
            for i, x in enumerate(lst):
                L = x.shape[0]
                stacked[i, :L] = x[:L]
            print(f'Warning: `{name}` had varying inner lengths {uniq}; padded to length {maxlen}.')
            return stacked
        raise ValueError(f'Unexpected shape/dtype for `{name}`: shape={getattr(arr, "shape", None)}, dtype={getattr(arr, "dtype", None)}')

    # Case A: interactions is label-only (N,) or (N,1)
    if getattr(interactions, 'ndim', None) == 1 or (getattr(interactions, 'ndim', None) == 2 and interactions.shape[1] == 1):
        # Normalize label array to (N,)
        if interactions.ndim == 2:
            labels = interactions[:, 0]
        else:
            labels = interactions
        # Ensure compounds/proteins are parallel arrays of length N
        if len(compounds) != len(proteins) or len(compounds) != len(labels):
            raise ValueError(f'Label-only interactions expected length N matching compounds/proteins. Found lengths: compounds={len(compounds)}, proteins={len(proteins)}, labels={len(labels)}')

        Xc = ensure_2d_numeric(compounds, 'compounds')
        Xp = ensure_2d_numeric(proteins, 'proteins')
        if Xc.shape[0] != Xp.shape[0] or Xc.shape[0] != len(labels):
            raise ValueError(f'After stacking, length mismatch: Xc.shape={Xc.shape}, Xp.shape={Xp.shape}, labels={len(labels)}')
        X = np.hstack([Xc, Xp])
        y = np.asarray(labels).astype(int).ravel()
        return X, y

    # Case B: interactions contains index triplets (N, >=3)
    # Normalize 1-D object arrays of tuples to 2-D numeric
    if interactions.ndim == 1:
        try:
            interactions = np.vstack(interactions)
        except Exception:
            try:
                interactions = np.array(list(interactions))
            except Exception:
                pass

    if interactions.ndim != 2 or interactions.shape[1] < 3:
        raise ValueError(f'interactions.npy must be either labels (N,) or index triples (N,3). Found shape={getattr(interactions, "shape", None)}, dtype={getattr(interactions, "dtype", None)}')

    c_idx = interactions[:, 0].astype(int)
    p_idx = interactions[:, 1].astype(int)
    y = interactions[:, 2].astype(int)

    Xc = ensure_2d_numeric(compounds, 'compounds')
    Xp = ensure_2d_numeric(proteins, 'proteins')

    try:
        X = np.hstack([Xc[c_idx], Xp[p_idx]])
    except Exception as e:
        raise IndexError(f'Indexing error mapping indices to features: {e}')

    return X, y

In [76]:
def train_and_evaluate(X, y, output_dir, dataset_name='dataset', test_size=0.2, random_state=0):
    os.makedirs(output_dir, exist_ok=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y if len(np.unique(y))>1 else None)

    models = {
        'knn': Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier())]),
        'rf': Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier(n_estimators=200, random_state=random_state))]),
        'svm': Pipeline([('scaler', StandardScaler()), ('clf', SVC(probability=True, random_state=random_state))]),
        'lr': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=random_state, solver='liblinear'))]),
        'gb': Pipeline([('scaler', StandardScaler()), ('clf', GradientBoostingClassifier(n_estimators=100, random_state=random_state))]),
        'ada': Pipeline([('scaler', StandardScaler()), ('clf', AdaBoostClassifier(n_estimators=100, random_state=random_state))]),
        'et': Pipeline([('scaler', StandardScaler()), ('clf', ExtraTreesClassifier(n_estimators=200, random_state=random_state))]),
        'nb': Pipeline([('scaler', StandardScaler()), ('clf', GaussianNB())]),
        'dummy': Pipeline([('scaler', StandardScaler()), ('clf', DummyClassifier(strategy='stratified', random_state=random_state))])
    }

    results = {}
    for name, model in models.items():
        print(f'Training {name}...')
        model.fit(X_train, y_train)
        # predict probabilities if available
        try:
            probs = model.predict_proba(X_test)[:,1]
        except Exception:
            # fallback to decision_function for SVM without prob (should be enabled above)
            try:
                probs = model.decision_function(X_test)
            except Exception:
                probs = None

        preds = model.predict(X_test)
        auc = roc_auc_score(y_test, probs) if (probs is not None and len(np.unique(y_test))>1) else float('nan')
        acc = accuracy_score(y_test, preds)
        prf = precision_recall_fscore_support(y_test, preds, average='binary', zero_division=0) if len(np.unique(y_test))<=2 else precision_recall_fscore_support(y_test, preds, average='macro', zero_division=0)

        results[name] = dict(auc=float(auc), accuracy=float(acc), precision=float(prf[0]), recall=float(prf[1]), f1=float(prf[2]))

        model_path = os.path.join(output_dir, f'classical_{dataset_name}_{name}.joblib')
        joblib.dump(model, model_path)
        print(f'saved model to {model_path}')

    # Save results summary
    results_path = os.path.join(output_dir, f'results_{dataset_name}.csv')
    df = pd.DataFrame.from_dict(results, orient='index')
    df.to_csv(results_path)
    print(f'saved results to {results_path}')
    return df

In [77]:
# Example runner: trains classical models for selected datasets
datasets = ['b_cancer']
base_dataset_dir = os.path.join('..', 'dataset') if os.path.exists(os.path.join('..','dataset')) else os.path.join(os.getcwd(), 'dataset')
# Adjust these input subpaths if your data uses a different radius/ngram folder
input_subpath = os.path.join('input', 'radius2_ngram3')
output_base = os.path.join('output', 'model_classical')

for ds in datasets:
    inp = os.path.join(base_dataset_dir, ds, input_subpath)
    if not os.path.exists(inp):
        print(f'WARNING: skipping {ds} because {inp} does not exist')
        continue
    try:
        X, y = load_interaction_dataset(inp)
    except Exception as e:
        print(f'ERROR loading {ds}: {e}')
        continue
    outdir = os.path.join(output_base, ds)
    df = train_and_evaluate(X, y, outdir, dataset_name=ds)
    print(df)

compounds.shape :  (1618,) proteins.shape :  (1618,) interactions.shape :  (1618, 1)
Training knn...
saved model to output\model_classical\b_cancer\classical_b_cancer_knn.joblib
Training rf...
saved model to output\model_classical\b_cancer\classical_b_cancer_knn.joblib
Training rf...
saved model to output\model_classical\b_cancer\classical_b_cancer_rf.joblib
Training svm...
saved model to output\model_classical\b_cancer\classical_b_cancer_rf.joblib
Training svm...
saved model to output\model_classical\b_cancer\classical_b_cancer_svm.joblib
Training lr...
saved model to output\model_classical\b_cancer\classical_b_cancer_svm.joblib
Training lr...
saved model to output\model_classical\b_cancer\classical_b_cancer_lr.joblib
Training gb...
saved model to output\model_classical\b_cancer\classical_b_cancer_lr.joblib
Training gb...
saved model to output\model_classical\b_cancer\classical_b_cancer_gb.joblib
Training ada...
saved model to output\model_classical\b_cancer\classical_b_cancer_gb.jobl

**Usage notes**:
- Run this notebook from the `ML/` folder or adapt the `base_dataset_dir` path in the runner cell.
- If your inputs are in a different `radius*/ngram*` folder, update `input_subpath`.
- Trained models are saved under `output/model_classical/<dataset>/classical_<dataset>_<model>.joblib`. Results summary saved as CSV in the same folder.

If you'd like, I can also: (a) add a CLI script `code/train_classicals.py`, (b) commit these changes, or (c) run the notebook here to validate outputs (I cannot run it on your machine without your confirmation).

In [78]:
# Diagnostic load: load raw object arrays (allow_pickle=True)
inp = os.path.join('..', 'dataset', 'b_cancer', 'input', 'radius2_ngram3')
compounds = np.load(os.path.join(inp, 'compounds.npy'), allow_pickle=True)
proteins = np.load(os.path.join(inp, 'proteins.npy'), allow_pickle=True)
interactions = np.load(os.path.join(inp, 'interactions.npy'), allow_pickle=True)
print('loaded raw arrays from', inp)

loaded raw arrays from ..\dataset\b_cancer\input\radius2_ngram3


In [79]:
# Show top-level shapes and dtypes
print('compounds.shape ->', getattr(compounds, 'shape', None), 'dtype ->', getattr(compounds, 'dtype', None))
print('proteins.shape   ->', getattr(proteins, 'shape', None), 'dtype ->', getattr(proteins, 'dtype', None))
print('interactions.shape ->', getattr(interactions, 'shape', None), 'dtype ->', getattr(interactions, 'dtype', None))

compounds.shape -> (1618,) dtype -> object
proteins.shape   -> (1618,) dtype -> object
interactions.shape -> (1618, 1) dtype -> int64


In [80]:
# Inspect a few sample inner-element types and lengths
def sample_inner_info(arr, n=5):
    for i in range(min(len(arr), n)):
        elem = arr[i]
        try:
            a = np.asarray(elem)
            print(f'[{i}] type={type(elem)}, ndim={getattr(a,ndim,None)}, shape={getattr(a,shape,None)}, dtype={getattr(a,dtype,None)}')
        except Exception as e:
            print(f'[{i}] could not convert element: {e}')

sample_inner_info(compounds)
sample_inner_info(proteins)

[0] could not convert element: name 'ndim' is not defined
[1] could not convert element: name 'ndim' is not defined
[2] could not convert element: name 'ndim' is not defined
[3] could not convert element: name 'ndim' is not defined
[4] could not convert element: name 'ndim' is not defined
[0] could not convert element: name 'ndim' is not defined
[1] could not convert element: name 'ndim' is not defined
[2] could not convert element: name 'ndim' is not defined
[3] could not convert element: name 'ndim' is not defined
[4] could not convert element: name 'ndim' is not defined


In [73]:
# # Optional: create cleaned/padded numeric versions and save them (non-destructive)
# # This produces `*_clean.npy` files next to the originals for faster reuse by classical models.
# def to_padded(arr, name=None):
#     lst = []
#     for i, e in enumerate(arr):
#         a = np.asarray(e)
#         if a.ndim == 0:
#             a = a.reshape(1)
#         lst.append(a.ravel())
#     lengths = [x.shape[0] for x in lst]
#     if len(set(lengths)) == 1:
#         return np.vstack(lst).astype(float)
#     maxlen = max(lengths)
#     out = np.zeros((len(lst), maxlen), dtype=float)
#     for i, x in enumerate(lst):
#         out[i, :x.shape[0]] = x
#     print(f'Warning: `{name}` padded to length {maxlen}')
#     return out

# inp = os.path.join('..', 'dataset', 'b_cancer', 'input', 'radius2_ngram3')
# c_clean_path = os.path.join(inp, 'compounds_clean.npy')
# p_clean_path = os.path.join(inp, 'proteins_clean.npy')
# i_clean_path = os.path.join(inp, 'interactions_clean.npy')
# if not (os.path.exists(c_clean_path) and os.path.exists(p_clean_path) and os.path.exists(i_clean_path)):
#     print('Creating cleaned padded arrays...')
#     C = to_padded(compounds, 'compounds')
#     P = to_padded(proteins, 'proteins')
#     I = np.asarray(interactions).ravel().astype(int)
#     np.save(c_clean_path, C)
#     np.save(p_clean_path, P)
#     np.save(i_clean_path, I)
#     print('Saved:', c_clean_path, p_clean_path, i_clean_path)
# else:
#     print('Cleaned files already exist:', c_clean_path)